# Certificados de conformidad con el Esquema Nacional de Seguridad

Cuaderno de lectura de la medición `ens-certificados/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Registro público de certificados vigentes del Centro Criptológico Nacional el 15 de octubre de 2026: categorías, tipos de entidad, certificadoras y vencimientos. Carga los ficheros de al lado (o los descarga del repositorio si se ejecuta fuera de él), muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no repite la medición: para eso está `generar.py`.

In [ ]:
import urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/ens-certificados/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
cert = leer("salidas/certificados.csv", dtype={"perfil": str})
print(len(cert), "certificados vigentes")
print(cert.groupby(["listado", "categoria"]).size().to_string())
print()
cat = cert[cert.listado == "categoria"]
print(cat.tipo_entidad.value_counts().to_string())
print()
cuotas = leer("salidas/certificadoras.csv")
total = cuotas.certificados.sum()
cuota = 100 * cuotas.certificados / total
print("certificadoras:", len(cuotas), "· cuota de las cuatro primeras:",
      round(cuota.iloc[:4].sum(), 1), "%")
print("índice de Herfindahl-Hirschman:", round((cuota ** 2).sum()))

## Una figura

In [ ]:
venc = pd.to_datetime(cert.fecha_vencimiento)
vigentes = cert[venc >= "2026-10-15"].copy()
vigentes["mes"] = pd.to_datetime(vigentes.fecha_vencimiento).dt.to_period("M")
tabla = vigentes.groupby(["mes", "listado"]).size().unstack(fill_value=0)
tabla = tabla.loc[:"2028-10"]
fig, ax = plt.subplots(figsize=(9, 4))
tabla.plot.bar(stacked=True, ax=ax, width=0.8)
ax.set_xticks(range(0, len(tabla), 3), [str(p) for p in tabla.index[::3]], rotation=0)
ax.set_ylabel("certificados que vencen")
ax.set_title("Vencimientos de los certificados del ENS vigentes el 15-10-2026")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()